In [ ]:
# 실행 환경과 데이터 경로 설정
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import nnls

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "conductive_paper_measurements_remeasured.csv"
OUTPUT_DIR = ROOT / "outputs" / "conductive_paper"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ELECTRODE_IDS = np.array([0, 1, 2, 3, 12, 13, 14, 15], dtype=int)
X_CM = np.array([4, 7, 10, 13, 16, 19, 22, 25], dtype=float)
Y_CM = np.array([3, 6, 9, 12, 15, 18], dtype=float)
POINT_COLUMNS = [f"P{i}" for i in range(48)]

plt.rcParams["axes.unicode_minus"] = False
print("데이터:", DATA_PATH)


In [ ]:
# 전위와 전기장 분석 함수 정의
def phi_to_electric_field(phi):
    grid = np.asarray(phi, dtype=float).reshape(len(Y_CM), len(X_CM))
    dx = float((X_CM[1] - X_CM[0]) * 1.0e-2)
    dy = float((Y_CM[1] - Y_CM[0]) * 1.0e-2)
    ex = -(grid[1:-1, 2:] - grid[1:-1, :-2]) / (2.0 * dx)
    ey = -(grid[2:, 1:-1] - grid[:-2, 1:-1]) / (2.0 * dy)
    return np.column_stack([ex.ravel(), ey.ravel()]).ravel()


def model_metrics(a_phi, y_phi, a_e, y_e, coefficients):
    predicted_phi = a_phi @ coefficients
    predicted_e = a_e @ coefficients
    true_vectors = y_e.reshape(-1, 2)
    predicted_vectors = predicted_e.reshape(-1, 2)
    e_rms = np.sqrt(np.mean(np.sum(true_vectors * true_vectors, axis=1))) + 1.0e-15
    local_error = np.sqrt(np.sum((predicted_vectors - true_vectors) ** 2, axis=1)) / e_rms
    return {
        "phi_relative_l2": float(
            np.linalg.norm(predicted_phi - y_phi) / (np.linalg.norm(y_phi) + 1.0e-15)
        ),
        "E_relative_l2": float(
            np.linalg.norm(predicted_e - y_e) / (np.linalg.norm(y_e) + 1.0e-15)
        ),
        "epsilon_E95": float(np.percentile(local_error, 95)),
    }


def fit_on_support(a_phi, y_phi, support):
    coefficients = np.zeros(a_phi.shape[1], dtype=float)
    coefficients[list(support)] = nnls(a_phi[:, list(support)], y_phi)[0]
    return coefficients


def positive_omp(a_phi, y_phi, k):
    residual = y_phi.copy()
    support = []
    norms = np.linalg.norm(a_phi, axis=0) + 1.0e-15
    for _ in range(k):
        correlations = np.maximum((a_phi.T @ residual) / norms, 0.0)
        if support:
            correlations[np.asarray(support, dtype=int)] = -np.inf
        support.append(int(np.argmax(correlations)))
        active = nnls(a_phi[:, support], y_phi)[0]
        residual = y_phi - a_phi[:, support] @ active
    return fit_on_support(a_phi, y_phi, tuple(support)), np.asarray(support, dtype=int)


def positive_lasso(a_phi, y_phi, alpha_ratio=0.10):
    norms = np.linalg.norm(a_phi, axis=0) + 1.0e-15
    normalized = a_phi / norms
    alpha = float(np.max(np.maximum(normalized.T @ y_phi, 0.0)) * alpha_ratio)
    weights = np.zeros(normalized.shape[1], dtype=float)
    residual = y_phi.copy()
    column_norms = np.sum(normalized * normalized, axis=0) + 1.0e-15
    for _ in range(5000):
        max_change = 0.0
        for j in range(normalized.shape[1]):
            residual += normalized[:, j] * weights[j]
            rho = float(normalized[:, j] @ residual)
            new_weight = max(rho - alpha, 0.0) / column_norms[j]
            residual -= normalized[:, j] * new_weight
            max_change = max(max_change, abs(new_weight - weights[j]))
            weights[j] = new_weight
        if max_change < 1.0e-9:
            break
    return weights / norms


def best_k_support(a_phi, y_phi, k):
    best = None
    for support in combinations(range(a_phi.shape[1]), k):
        coefficients = fit_on_support(a_phi, y_phi, support)
        error = np.linalg.norm(a_phi @ coefficients - y_phi) / (np.linalg.norm(y_phi) + 1.0e-15)
        if best is None or error < best[0]:
            best = (float(error), coefficients, np.asarray(support, dtype=int))
    return best[1], best[2]


In [ ]:
# 재측정 데이터와 응답 행렬 생성
df = pd.read_csv(DATA_PATH)
offset = df.loc[df["run"] == "Z0", POINT_COLUMNS].iloc[0].to_numpy(float)

dictionary_columns = []
drive_voltage = {}
for electrode_id in ELECTRODE_IDS:
    row = df.loc[df["run"] == f"A{electrode_id}"].iloc[0]
    drive_voltage[int(electrode_id)] = float(row["drive_voltage"])
    dictionary_columns.append(
        (row[POINT_COLUMNS].to_numpy(float) - offset) / float(row["drive_voltage"])
    )

a_phi = np.column_stack(dictionary_columns)
y_phi = (
    df.loc[df["run"] == "T_E2_E13", POINT_COLUMNS].iloc[0].to_numpy(float) - offset
)
a_e = np.column_stack(
    [phi_to_electric_field(a_phi[:, j]) for j in range(a_phi.shape[1])]
)
y_e = phi_to_electric_field(y_phi)

print("전위 응답 행렬:", a_phi.shape)
print("전기장 응답 행렬:", a_e.shape)


In [ ]:
# OMP, Lasso, NNLS 피팅
local_index = {int(eid): i for i, eid in enumerate(ELECTRODE_IDS)}
true_support = np.array([local_index[2], local_index[13]], dtype=int)

direct_coefficients = np.zeros(len(ELECTRODE_IDS))
direct_coefficients[true_support] = [drive_voltage[2], drive_voltage[13]]

omp_coefficients, omp_support = positive_omp(a_phi, y_phi, 2)
lasso_coefficients = positive_lasso(a_phi, y_phi)
best3_coefficients, best3_support = best_k_support(a_phi, y_phi, 3)
dense_coefficients = nnls(a_phi, y_phi)[0]

model_inputs = [
    ("Direct E2+E13", direct_coefficients, true_support),
    ("OMP K=2 + NNLS", omp_coefficients, omp_support),
    ("Positive Lasso", lasso_coefficients, np.flatnonzero(lasso_coefficients > 1.0e-6)),
    ("Best K=3 NNLS", best3_coefficients, best3_support),
    ("Dense NNLS", dense_coefficients, np.flatnonzero(dense_coefficients > 1.0e-6)),
]

model_rows = []
for method, coefficients, support in model_inputs:
    metrics = model_metrics(a_phi, y_phi, a_e, y_e, coefficients)
    model_rows.append(
        {
            "method": method,
            "support": [int(x) for x in ELECTRODE_IDS[np.asarray(support, dtype=int)]],
            "coefficients": coefficients,
            **metrics,
        }
    )

normalized = a_phi / (np.linalg.norm(a_phi, axis=0, keepdims=True) + 1.0e-15)
gram = np.abs(normalized.T @ normalized)
np.fill_diagonal(gram, 0.0)
true_pair_coherence = float(gram[true_support[0], true_support[1]])
true_pair_condition = float(np.linalg.cond(a_phi[:, true_support]))

result_table = pd.DataFrame(
    [
        {
            "method": row["method"],
            "support": row["support"],
            "phi_relative_l2": row["phi_relative_l2"],
            "E_relative_l2": row["E_relative_l2"],
            "epsilon_E95": row["epsilon_E95"],
        }
        for row in model_rows
    ]
)

print("Mutual coherence:", float(np.max(gram)))
print("E2-E13 coherence:", true_pair_coherence)
print("E2-E13 condition number:", true_pair_condition)
display(result_table)


In [ ]:
# 측정 잡음에 대한 지지집합 복원 확인
rng = np.random.default_rng(2026)
recovery_rows = []
true_set = {2, 13}

for noise_fraction in [0.0, 0.002, 0.005, 0.01, 0.02, 0.05]:
    sigma = noise_fraction * float(np.sqrt(np.mean(y_phi**2)))
    omp_success = 0
    best2_success = 0
    trials = 80
    for _ in range(trials):
        noisy_target = y_phi + rng.normal(0.0, sigma, size=y_phi.shape)
        _, omp_local = positive_omp(a_phi, noisy_target, 2)
        _, best2_local = best_k_support(a_phi, noisy_target, 2)
        omp_success += int(set(map(int, ELECTRODE_IDS[omp_local])) == true_set)
        best2_success += int(set(map(int, ELECTRODE_IDS[best2_local])) == true_set)
    recovery_rows.append(
        {
            "noise_fraction": noise_fraction,
            "OMP": omp_success / trials,
            "best_2_NNLS": best2_success / trials,
        }
    )

display(pd.DataFrame(recovery_rows))


In [ ]:
# 전위 피팅과 모델 오차 시각화
omp_row = next(row for row in model_rows if row["method"] == "OMP K=2 + NNLS")
fitted_phi = a_phi @ omp_row["coefficients"]
target_grid = y_phi.reshape(len(Y_CM), len(X_CM))
fitted_grid = fitted_phi.reshape(len(Y_CM), len(X_CM))
residual_grid = target_grid - fitted_grid

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, values, title, cmap in zip(
    axes,
    [target_grid, fitted_grid, residual_grid],
    ["Measured target", "OMP + NNLS fit", "Residual"],
    ["viridis", "viridis", "RdBu_r"],
):
    image = ax.imshow(
        values,
        origin="lower",
        extent=[X_CM.min(), X_CM.max(), Y_CM.min(), Y_CM.max()],
        aspect="auto",
        cmap=cmap,
    )
    ax.set_title(title)
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "potential_fit.png", dpi=180)
plt.show()

labels = [row["method"] for row in model_rows]
phi_errors = [row["phi_relative_l2"] for row in model_rows]
field_errors = [row["E_relative_l2"] for row in model_rows]
xpos = np.arange(len(labels))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(xpos - width / 2, phi_errors, width, label="Potential L2")
ax.bar(xpos + width / 2, field_errors, width, label="Electric-field L2")
ax.set_xticks(xpos)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Relative L2 error")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_errors.png", dpi=180)
plt.show()
